In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Performance Optimization Demo") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

In [ ]:
employees = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/employees.csv")

In [ ]:
departments = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/departments.csv")

In [ ]:
log_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("ip", StringType(), True),
    StructField("method", StringType(), True),
    StructField("url", StringType(), True),
    StructField("status", IntegerType(), True),
    StructField("response_time", IntegerType(), True),
    StructField("service", StringType(), True)
])

In [23]:
logs = spark.readStream \
    .schema(log_schema) \
    .option("header", True) \
    .csv("data/logs/")

In [24]:
logs.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- ip: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- response_time: integer (nullable = true)
 |-- service: string (nullable = true)



In [25]:
processed_logs = logs.withColumn(
    "status_category",
    when(col("status") < 300, "SUCCESS")
    .when(col("status") < 400, "REDIRECTION")
    .when(col("status") < 500, "CLIENT_ERROR")
    .otherwise("SERVER_ERROR")
)
processed_logs.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- ip: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- response_time: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- status_category: string (nullable = false)



In [26]:
processed_logs = processed_logs.withColumn(
    "performance_category",
    when(
        col("response_time") > 1000,
        "SLOW"
    ).otherwise("NORMAL")
)
processed_logs.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- ip: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- response_time: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- status_category: string (nullable = false)
 |-- performance_category: string (nullable = false)



In [27]:
valid_logs = processed_logs.filter(
    col("timestamp").isNotNull()
    & col("url").isNotNull()
    & col("status").isNotNull()
    & (col("response_time") >= 0)
)

In [28]:
windowed_logs = valid_logs.withWatermark(
    "timestamp",
    "2 minutes"
)

In [29]:
metrics = windowed_logs.groupBy(
    window("timestamp", "5 minutes")
).agg(
    count("*").alias("total_requests"),
    sum(
        when(col("status") < 400, 1).otherwise(0)
    ).alias("successful_requests"),
    sum(
        when(col("status") >= 400, 1).otherwise(0)
    ).alias("failed_requests"),
    round(
        avg("response_time"),
        2
    ).alias("avg_response_time"),
    sum(
        when(col("response_time") > 1000, 1)
        .otherwise(0)
    ).alias("slow_requests")
)

In [31]:
query = metrics.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("metrics2") \
    .start()

In [21]:
query.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [32]:
spark.sql("""
    SELECT *
    FROM metrics2
""").show(truncate=False)

+------------------------------------------+--------------+-------------------+---------------+-----------------+-------------+
|window                                    |total_requests|successful_requests|failed_requests|avg_response_time|slow_requests|
+------------------------------------------+--------------+-------------------+---------------+-----------------+-------------+
|{2026-08-19 10:00:00, 2026-08-19 10:05:00}|10            |6                  |4              |489.0            |2            |
+------------------------------------------+--------------+-------------------+---------------+-----------------+-------------+



In [ ]:
endpoint_metrics = valid_logs.withWatermark(
    "timestamp",
    "2 minutes"
).groupBy(
    window("timestamp", "5 minutes"),
    "url"
).agg(
    count("*").alias("requests"),
    round(avg("response_time"), 2).alias("avg_response_time"),
    sum(
        when(col("status") >= 400, 1).otherwise(0)
    ).alias("errors")
)

In [33]:
query.stop()